# Grover Search and Amplitude Amplification — Teaching Notebook

Grover’s algorithm searches an unstructured space by rotating amplitude toward marked states. It provides a quadratic query advantage and is one of the clearest demonstrations of constructive/destructive interference.

This notebook is designed to be **self-explanatory for classroom teaching**. It moves from the problem statement and mathematics to quantum circuits, Qiskit implementation, interpretation, limitations, and exercises.

### Learning objectives
- derive the phase oracle and diffuser
- understand Grover as a 2D rotation
- calculate the optimal iteration count
- implement single- and multi-solution search
- distinguish Grover search from general amplitude amplification

## 0. Installation
Run this only if Qiskit is not already installed.

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib numpy

## 1. Imports and helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, pi

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

backend = AerSimulator()

def run_counts(qc, shots=2048):
    compiled = transpile(qc, backend)
    return backend.run(compiled, shots=shots).result().get_counts()

def plot_counts(counts, title='Measurement counts'):
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    plt.figure(figsize=(8,4))
    plt.bar(keys, vals)
    plt.xlabel('bit string')
    plt.ylabel('counts')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

# Part I — Search problem
Suppose there are $N=2^n$ possible items and an oracle marks $M$ solutions. Classical unstructured search requires $O(N/M)$ oracle queries on average/worst-case scale. Grover requires

$$O\left(\sqrt{N/M}\right)$$

oracle uses.

## 2. Uniform superposition
Apply Hadamards to $n$ qubits:

$$|s\rangle=H^{\otimes n}|0^n\rangle=\frac1{\sqrt N}\sum_{x=0}^{N-1}|x\rangle.$$

Every basis state initially has the same amplitude $1/\sqrt N$.

# Part II — Phase oracle
For a Boolean predicate $f(x)$, a Grover phase oracle acts as

$$O_f|x\rangle=(-1)^{f(x)}|x\rangle.$$

Marked states get a minus sign; unmarked states are unchanged. The probabilities have not changed yet — only phases have.

In [ ]:
def mark_bitstring(qc, qubits, target):
    # target is written in q_{n-1}...q_0 display order, e.g. '101'
    bits=[int(b) for b in target[::-1]]
    for q,b in zip(qubits,bits):
        if b==0: qc.x(q)
    if len(qubits)==1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        qc.mcx(qubits[:-1],qubits[-1])
        qc.h(qubits[-1])
    for q,b in zip(qubits,bits):
        if b==0: qc.x(q)

## 3. Diffusion operator
The diffuser is reflection about the uniform state:

$$D=2|s\rangle\langle s|-I.$$

Circuit identity:

$$D=H^{\otimes n}X^{\otimes n}\,(2|0^n\rangle\langle0^n|-I)\,X^{\otimes n}H^{\otimes n}.$$

Up to an irrelevant global phase, the central reflection can be implemented using a multi-controlled $Z$.

In [ ]:
def diffuser(qc, qubits):
    qc.h(qubits)
    qc.x(qubits)
    if len(qubits)==1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        qc.mcx(qubits[:-1],qubits[-1])
        qc.h(qubits[-1])
    qc.x(qubits)
    qc.h(qubits)

# Part III — One complete Grover iteration

In [ ]:
def grover_search(n,target,iterations=1):
    qc=QuantumCircuit(n,n)
    qs=list(range(n))
    qc.h(qs)
    for _ in range(iterations):
        mark_bitstring(qc,qs,target)
        diffuser(qc,qs)
    qc.measure(qs,qs)
    return qc

qc=grover_search(3,'101',1)
print(qc.draw(fold=120))
print(run_counts(qc))

## 4. Geometric derivation
Let

$$|\beta\rangle=\frac1{\sqrt M}\sum_{x\in\text{marked}}|x\rangle,$$

and $|\alpha\rangle$ be the normalized superposition of unmarked states. Then

$$|s\rangle=\cos\theta|\alpha\rangle+\sin\theta|\beta\rangle,$$

with

$$\sin^2\theta=\frac MN.$$

Each Grover iteration rotates the state by approximately $2\theta$ toward $|\beta\rangle$. After $r$ iterations,

$$P_{\rm success}=\sin^2((2r+1)\theta).$$

## 5. Optimal number of iterations
For $M\ll N$,

$$r_{\rm opt}\approx\left\lfloor\frac\pi4\sqrt{\frac NM}\right\rfloor.$$

Too many iterations **overshoot** the marked subspace and reduce success probability.

In [ ]:
def grover_success_theory(N,M,r):
    theta=np.arcsin(np.sqrt(M/N))
    return np.sin((2*r+1)*theta)**2

N=32; M=1
for r in range(8):
    print(r, round(grover_success_theory(N,M,r),4))

# Part IV — Visualize amplitude amplification

In [ ]:
N=32; M=1
rs=np.arange(0,10)
ps=[grover_success_theory(N,M,int(r)) for r in rs]
plt.figure(figsize=(7,4))
plt.plot(rs,ps,'o-')
plt.xlabel('Grover iterations')
plt.ylabel('success probability')
plt.title('Grover oscillation: amplification then overshoot')
plt.ylim(0,1.05)
plt.grid(True,alpha=.3)
plt.show()

## 6. Three-qubit example at the optimal iteration count

In [ ]:
n=3; N=2**n; M=1
r=int(np.floor(np.pi/4*np.sqrt(N/M)))
print('iterations =',r)
counts=run_counts(grover_search(n,'101',r),4096)
print(counts)
plot_counts(counts,'Grover search for |101>')

# Part V — Multiple marked states
When $M>1$, the oracle flips every marked state. The same geometry applies, but $\sin^2\theta=M/N$ and fewer iterations are needed.

In [ ]:
def grover_multiple(n,targets,iterations):
    qc=QuantumCircuit(n,n); qs=list(range(n)); qc.h(qs)
    for _ in range(iterations):
        for t in targets: mark_bitstring(qc,qs,t)
        diffuser(qc,qs)
    qc.measure(qs,qs)
    return qc

n=4; targets=['0011','1100']; M=len(targets); N=2**n
r=int(np.floor(np.pi/4*np.sqrt(N/M)))
print('iterations',r)
counts=run_counts(grover_multiple(n,targets,r),4096)
plot_counts(counts,'Two marked states')

# Part VI — Mean inversion intuition
For one marked state, the oracle makes its amplitude negative. The diffuser then maps every amplitude $a_i$ to

$$a_i' = 2\bar a-a_i,$$

where $\bar a$ is the average amplitude. A negative marked amplitude far below the mean is reflected to a large positive value, while unmarked amplitudes decrease slightly.

In [ ]:
amps=np.ones(8)/np.sqrt(8)
amps[5]*=-1
mean=amps.mean()
new=2*mean-amps
print('after oracle:',np.round(amps,3))
print('mean:',round(mean,3))
print('after inversion about mean:',np.round(new,3))
print('probability marked:',round(new[5]**2,3))

# Part VII — Grover versus amplitude amplification
Grover assumes the initial state is the uniform superposition. **Amplitude amplification** generalizes the idea to any state-preparation operator $A|0\rangle$ with some initial probability of landing in a “good” subspace. The two reflections amplify that good component.

# Part VIII — Limits
Grover gives a **quadratic**, not exponential, speedup. The oracle must be implemented coherently, and its gate cost counts in a real algorithm. For structured problems, classical or quantum algorithms exploiting that structure may beat generic Grover search.

# Part IX — Exercises
1. For $N=64,M=1$, estimate the optimal iteration count.
2. Derive $P=\sin^2((2r+1)\theta)$ from the 2D rotation picture.
3. Modify the code to mark 0110 in a 4-qubit search.
4. Mark two states and compare the optimal $r$ with the single-target case.
5. Explain why the oracle alone cannot increase the success probability.

# Part X — Solutions
For $N=64,M=1$, $r\approx\lfloor(\pi/4)8\rfloor=6$. The oracle changes phase but not amplitude magnitude, so measurement probabilities immediately after the oracle are unchanged. Amplification appears only after interference through the diffuser.

# Compact reference
$$O_f|x\rangle=(-1)^{f(x)}|x\rangle$$

$$D=2|s\rangle\langle s|-I$$

$$G=DO_f$$

$$\sin^2\theta=M/N,\qquad P_r=\sin^2((2r+1)\theta),\qquad r\approx\frac\pi4\sqrt{N/M}.